# SwinIR 单通道灰度版微调

GPU: 2×T4 | 从零训练单通道模型

**策略**：
1. datasets/train 已经是灰度图，直接生成 LR 配对
2. 创建单通道 SwinIR (in_chans=1)，从零训练
3. 中间 Transformer layers (180→180) 完全复用 RGB 版架构
4. 70k iter，2×T4 预计 ~9.6h

## 0. 环境准备

In [ ]:
import torch
NUM_GPUS = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {NUM_GPUS}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0))
    print(f'  GPU {i}: {p.name} — {mem/1e9:.1f} GB')

SEC_PER_ITER = 0.982  # 与 RGB 版相同
MAX_SECONDS = 11.5 * 3600
MAX_ITERS = int(MAX_SECONDS / SEC_PER_ITER / NUM_GPUS) if NUM_GPUS > 0 else 40000
TOTAL_ITER = MAX_ITERS
print(f'\n预估: {NUM_GPUS}×GPU, ~{SEC_PER_ITER}s/iter → total_iter={TOTAL_ITER}')

In [ ]:
import os, shutil
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)

# 克隆仓库
if not os.path.exists('BasicSR'):
    !git clone https://github.com/Log-Dog012/BasicSR.git
    !cd BasicSR && git checkout cuda-sem-finetune

# 安装依赖
os.chdir('BasicSR')
!pip install -r requirements.txt -q
!pip install -e . -q
!pip install lpips timm -q
print('✅ 环境就绪')

# 复制 checkpoint（如果有 RGB 版微调权重做参考）
MODEL_INPUT = '/kaggle/input/models/logdog012/swinir-finetune/pytorch/default/2'
for root, dirs, files in os.walk(MODEL_INPUT):
    for f in files:
        if f.endswith('.pth') and 'net_g' in f:
            dst = os.path.join(os.getcwd(), 'experiments', 'pretrained_models', f)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(os.path.join(root, f), dst)
            print(f'✅ {f} → pretrained_models/')

## 1. 准备灰度训练数据

datasets/train 已经是灰度 SEM 图，直接降采样生成 LR 配对

In [ ]:
import cv2
import numpy as np

DATA_INPUT = '/kaggle/input/datasets/logdog012/swinir-finetune'
MODEL_INPUT = '/kaggle/input/models/logdog012/swinir-finetune/pytorch/default/2'

# Kaggle 数据集已包含灰度 HR/LR 配对，直接链接到 BasicSR 目录结构
# 训练数据
kaggle_hr = os.path.join(DATA_INPUT, 'train_sem_hr', 'HR')
kaggle_lr = os.path.join(DATA_INPUT, 'train_sem_lr_x4', 'LR_x4')

# 在 working 中创建链接（BasicSR 配置指向这里）
gray_root = os.path.join(WORK_DIR, 'datasets', 'swinir_train_gray')
hr_gray = os.path.join(gray_root, 'HR')
lr_gray = os.path.join(gray_root, 'LR_x4')

if not os.path.exists(hr_gray):
    os.makedirs(os.path.dirname(hr_gray), exist_ok=True)
    os.symlink(kaggle_hr, hr_gray)
    os.symlink(kaggle_lr, lr_gray)
    print(f'✅ 训练数据链接: {gray_root} → {DATA_INPUT}')
else:
    print(f'✅ 训练数据已存在: {gray_root}')

hr_count = len([f for f in os.listdir(hr_gray) if f.lower().endswith(('.jpg','.png'))])
lr_count = len([f for f in os.listdir(lr_gray) if f.lower().endswith(('.jpg','.png'))])
print(f'  HR: {hr_count} 张, LR: {lr_count} 张')

# eval 数据
kaggle_eval_hr = os.path.join(DATA_INPUT, 'eval_sem', 'eval', 'SEMimg')
kaggle_eval_lr = os.path.join(DATA_INPUT, 'eval_sem', 'eval', 'LR')

eval_hr_gray = os.path.join(WORK_DIR, 'datasets', 'eval', 'SEMimg_gray')
eval_lr_gray = os.path.join(WORK_DIR, 'datasets', 'eval', 'LR_gray')

if not os.path.exists(eval_hr_gray):
    os.makedirs(os.path.dirname(eval_hr_gray), exist_ok=True)
    os.symlink(kaggle_eval_hr, eval_hr_gray)
    os.symlink(kaggle_eval_lr, eval_lr_gray)
    print(f'✅ eval 数据链接: eval → {DATA_INPUT}')
else:
    print(f'✅ eval 数据已存在')

eval_count = len([f for f in os.listdir(eval_hr_gray) if f.lower().endswith(('.jpg','.png'))])
print(f'  eval: {eval_count} 张')

## 2. 训练

单通道 SwinIR 从零训练，70k iter

In [ ]:
# 复制配置文件
import subprocess

opt_path = 'options/train/SwinIR/finetune_SwinIR_SRx4_SEM_gray.yml'

print(f'开始训练 ({TOTAL_ITER} iter, {NUM_GPUS}×GPU)...')
print('='*50)

if NUM_GPUS >= 2:
    !torchrun --nproc_per_node={NUM_GPUS} --master_port=4321 -m basicsr.train \
      -opt {opt_path} \
      --launcher pytorch --auto_resume \
      --force_yml train:total_iter={TOTAL_ITER}
else:
    !python -m basicsr.train \
      -opt {opt_path} \
      --auto_resume \
      --force_yml train:total_iter={TOTAL_ITER}

## 3. 结果分析

In [ ]:
import re

exp_dir = os.path.join(WORK_DIR, 'BasicSR', 'experiments', 'finetune_SwinIR_SRx4_SEM_gray')
logs = sorted([f for f in os.listdir(exp_dir) if f.endswith('.log')])

if logs:
    latest_log = os.path.join(exp_dir, logs[-1])
    with open(latest_log, 'r') as f:
        content = f.read()
    
    # 提取验证 PSNR
    val_matches = re.findall(r'psnr:\s+([\d.]+)\s+Best:\s+([\d.]+)\s+@\s+(\d+)', content)
    
    if val_matches:
        print('验证结果:')
        print(f'{"Iter":<10} {"PSNR":<10} {"Best PSNR":<12}')
        print('-' * 35)
        best_psnr = 0
        for psnr_val, best_val, iter_val in val_matches:
            marker = ' ←' if float(psnr_val) == float(best_val) else ''
            print(f'{iter_val:<10} {psnr_val:<10} {best_val:<12}{marker}')
    
    # 最后几行
    lines = content.strip().split('\n')
    print(f'\n日志最后 5 行:')
    for line in lines[-5:]:
        print(line)
else:
    print('未找到日志')

## 4. 保存结果

In [ ]:
models_dir = os.path.join(exp_dir, 'models')
states_dir = os.path.join(exp_dir, 'training_states')

print('训练产物:')
print(f'  模型: {models_dir}')
if os.path.exists(models_dir):
    for f in sorted(os.listdir(models_dir)):
        size = os.path.getsize(os.path.join(models_dir, f)) / 1e6
        print(f'    {f} ({size:.1f}MB)')

print(f'\n  状态: {states_dir}')
if os.path.exists(states_dir):
    for f in sorted(os.listdir(states_dir)):
        size = os.path.getsize(os.path.join(states_dir, f)) / 1e6
        print(f'    {f} ({size:.1f}MB)')

# 汇总对比表
print('\n' + '='*60)
print('📊 单通道灰度实验结果')
print('='*60)
print('灰度模型 vs RGB 参考:')
print(f'  RGB 版 15k 最佳:  PSNR=26.64  SSIM=0.6794  LPIPS=0.3717')
print(f'  Bicubic 基线:     PSNR=25.38  SSIM=0.6480  LPIPS=0.5918')
print('='*60)
print('Save and Run All 后，右侧面板 → Data → 下载。')